In [ ]:
!pip install -q ddgs ollama
!apt-get install -y zstd -qq
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q statsmodels


!pkill -f "ollama serve"
import time
time.sleep(2)

import subprocess, time

with open('out/ollama.log', 'w') as log_file:
    subprocess.Popen(['ollama', 'serve'], stdout=log_file, stderr=log_file)

time.sleep(5)
print("Service Ollama demarre")

# Gemma 3
#!ollama pull gemma3:1b
!ollama pull gemma3:4b
#!ollama pull gemma3:12b

# Qwen 2.5
#!ollama pull qwen2.5:1.5b
!ollama pull qwen2.5:7b
#!ollama pull qwen2.5:14b

# Llama 3.2
!ollama pull llama3.2:1b
!ollama pull llama3.2:3b

# Phi-4
!ollama pull phi4-mini


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import csv
import random
import re
import time
import ollama
from ddgs import DDGS
from collections import Counter
from statsmodels.stats.contingency_tables import cochrans_q
from scipy.stats import chi2
import numpy as np
import os, datetime
from urllib.parse import urlparse

In [ ]:
!nvidia-smi

In [ ]:
def lire_csv(chemin):
    with open(chemin, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

In [ ]:
chemin_in = "data/"
chemin_cat_sources = "data/"
chemin_tirage = "data/tirages_7.csv"

chemin_out = "out/"

articles = lire_csv(chemin_in + "echantillon_150.csv")

catA = lire_csv(chemin_cat_sources + "domaines_A.csv")
catB = lire_csv(chemin_cat_sources + "domaines_B.csv")
catC = lire_csv(chemin_cat_sources + "domaines_C.csv")
catD = lire_csv(chemin_cat_sources + "domaines_D.csv")

print(f"Echantillon  : {len(articles)} articles")
print(f"Cat A  : {len(catA)} domaines")
print(f"Cat B  : {len(catB)} domaines")
print(f"Cat C  : {len(catC)} domaines")
print(f"Cat D  : {len(catD)} domaines")


In [ ]:
# pour avoir les 7 tirages 
random.seed(42)

N = 7  # nombre de tirages = nombre de domaines de B

domB = [d["domaine_principal"] for d in catB]
domA = [d["domaine_principal"] for d in catA]
domC = [d["domaine_principal"] for d in catC]
domD = [d["domaine_principal"] for d in catD]

assert len(domB) == N, f"B contient {len(domB)} domaines, pas {N}"

random.shuffle(domB)
tirA = random.sample(domA, N)
tirC = random.sample(domC, N)
tirD = random.sample(domD, N)

TIRAGES = [{"A": tirA[i], "B": domB[i], "C": tirC[i], "D": tirD[i]} for i in range(N)]

with open(chemin_out + "tirages_7.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["tirage", "A", "B", "C", "D"])
    w.writeheader()
    for i, t in enumerate(TIRAGES, 1):
        w.writerow({"tirage": i, **t})

for i, t in enumerate(TIRAGES, 1):
    print(i, t)

tous = [d for t in TIRAGES for d in t.values()]
assert len(tous) == 28, f"{len(tous)} domaines au lieu de 28"
assert len(set(tous)) == 28, "il y a des domaines répétés"
print("OK : 7 tirages x 4 domaines = 28 domaines tous différents")

In [ ]:
CHEMIN_PREUVES = "data/preuves_150-2.csv"

lignes_preuves = lire_csv(CHEMIN_PREUVES)

preuves_csv = {r["id"]: r["Preuves"] for r in lignes_preuves}

for a in articles:
    a["preuves"] = preuves_csv[a["id"]]

print(len(lignes_preuves), "lignes lues,", len(preuves_csv), "preuves rechargées")

In [ ]:
def chercher_preuves(requete):
    # cherche sur web des infos sur le texte passé en param
    # retourne les 3 premiers resultats trouvés formatés en liste de points

    #requete = titre if titre else texte[:100]
    try:
        resultats = DDGS().text(requete, max_results=3)
    except Exception as e:
        print(f"  Erreur de recherche : {e}")
        return "(aucune preuve trouvee)"

    if not resultats:
        return "(aucune preuve trouvee)"

    morceaux = []
    for r in resultats:
        morceaux.append(f"- {r.get('title', '')} : {r.get('body', '')[:200]}")
    return "\n".join(morceaux)


In [ ]:
def chercher_preuves_2(requete, max_tentatives=3):
    # cherche sur web des infos sur le texte passé en param
    # retourne les 3 premiers resultats trouvés formatés en liste de points
    # difference avec l'autre, si on échoue a trouver des preuves, on essaye jusqu'a 3 tentatives


    for tentative in range(1, max_tentatives + 1):
        try:
            # timeout augmente a 20s (au lieu de 5s par defaut)
            with DDGS(timeout=20) as ddgs:
                resultats = ddgs.text(query=requete, max_results=3)

            if not resultats:
                return "(aucune preuve trouvee)", []

            morceaux = []
            domaines = []
            
            for r in resultats:
                morceaux.append(f"- {r.get('title', '')} : {r.get('body', '')[:200]}")
                domaines.append(urlparse(r.get("href", "")).netloc)
            return "\n".join(morceaux), domaines

        except Exception as e:
            print(f"  [!] Tentative {tentative}/{max_tentatives} echouee : {e}")
            if tentative < max_tentatives:
                # attend de plus en plus longtemps a chaque tentative (3s, 6s, 9s)
                time.sleep(3 * tentative)

    # si toutes les tentatives ont echoue
    return "(aucune preuve trouvee apres plusieurs tentatives)", []


In [ ]:
def construire_prompt_fr(titre, texte, source, preuves, avec_rag):
    texte_court = texte[:800]

    prompt = "Voici un article a verifier.\n\n"
    if source != "":
        prompt = prompt + "Source : " + source + "\n"
    prompt = prompt + "Titre : " + titre + "\n"
    prompt = prompt + "Texte : " + texte_court + "\n\n"

    if avec_rag:
        prompt = prompt + "Preuves trouvees sur le web concernant cette affirmation :\n"
        prompt = prompt + preuves + "\n\n"
        prompt = prompt + "En te basant sur ces preuves, verifie si l'affirmation de cet article est vraie ou fausse.\n"
    else:
        prompt = prompt + "En te basant sur ce que tu connais déja, verifie si l'affirmation de cet article est vraie ou fausse.\n"
    
    prompt = prompt + "Reponds uniquement par un seul mot : \"fake\" ou \"real\"."

    return prompt


def construire_prompt(titre, texte, source, preuves, avec_rag):
    texte_court = texte[:2000]

    prompt = "Here is an article to verify.\n\n"
    if source != "":
        prompt = prompt + "Source: " + source + "\n"
    prompt = prompt + "Title: " + titre + "\n"
    prompt = prompt + "Text: " + texte_court + "\n\n"

    if avec_rag:
        prompt = prompt + "Evidence found on the web about this claim:\n"
        prompt = prompt + preuves + "\n\n"
        prompt = prompt + "Based on this evidence, verify whether the claim in this article is true or false.\n"
    else:
        prompt = prompt + "Based on what you already know, verify whether the claim in this article is true or false.\n"

    prompt = prompt + "Reply with one word only: \"fake\" or \"real\"."

    return prompt


def extraire_verdict(reponse_llm):
    texte = reponse_llm.lower().strip()

    contient_fake = bool(re.search(r"\bfake\b", texte))
    contient_real = bool(re.search(r"\breal\b", texte))

    # uniquemnt fake
    if contient_fake and not contient_real:
        return "fake"

    # uniquemnt real
    if contient_real and not contient_fake:
        return "real"

    # sinon
    return "indetermine"

In [ ]:
def demander_verdict(titre, texte, source, preuves, avec_rag):

    prompt = construire_prompt(titre, texte, source, preuves, avec_rag)

    #reponse = ollama.chat(model=m, messages=[{"role": "user", "content": prompt}])

    reponse = ollama.chat(
        model=m,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0}
    )

    contenu = reponse["message"]["content"]

    return extraire_verdict(contenu)


In [ ]:
FICHIER_PREUVES = chemin_out + "preuves_150.csv"

def sauver_preuve(id_article, requete, preuves, domaines):
    nouveau = not os.path.exists(FICHIER_PREUVES)
    with open(FICHIER_PREUVES, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if nouveau:
            w.writerow(["id", "requete", "Preuves", "domaine", "date_recherche"])
        w.writerow([id_article, requete, preuves, " | ".join(domaines), datetime.date.today().isoformat()])

In [ ]:
def tester_contrefactuel(article, source, avec_rag, creer_preuves):

    titre = article.get("title", "")
    texte = article.get("text", "")
    vraie_source = article.get("domaine", "source inconnue")
    vrai_label = article.get("label", "")

    # RAG ou non-RAG
    if avec_rag:
        if creer_preuves:
            requete = titre if titre else texte[:100]
            preuves, domaines_rag = chercher_preuves_2(requete)

            # Sauvegarde en memoire
            article["preuves"] = preuves

            # Sauvegarde dans le fichier CSV
            sauver_preuve(article["id"], requete, preuves, domaines_rag)
            print(f"RAG CRÉÉ pour : {article.get('id', '')}")

        else:
            # reutilisation des memes preuves
            preuves = article["preuves"]
            print(f"RAG RÉUTILISÉ pour : {article.get('id', '')}")

    else:
        preuves = None

    # Un seul verdict, avec la source pass en paramètre
    verdict = demander_verdict(titre, texte, source, preuves, avec_rag)

    return {
        "id": article.get("id", ""),
        "vraie_source": vraie_source,
        "vrai_label": vrai_label,
        "source_testee": source,
        "verdict": verdict
    }

In [ ]:
tirage = [{k: r[k] for k in "ABCD"} for r in lire_csv(chemin_tirage)]

for i, t in enumerate(tirage, 1):
    print(i, t)

print(len(tirage), "tirages chargés")

In [ ]:
def lancer_test_tirage(echantillon, chemin_sortie, sources, avec_rag):
    colonnes = ["id", "vraie_source", "vrai_label",
                "source_A", "verdict_A", "source_B", "verdict_B",
                "source_C", "verdict_C", "source_D", "verdict_D",
                "verdict_S"]

    lignes = []
    for i, article in enumerate(echantillon, 1):
        ligne = {
            "id": article.get("id"),
            "vraie_source": article.get("domaine", "source inconnue"),
            "vrai_label": article.get("label"),
        }

        ligne["verdict_S"] = tester_contrefactuel(article, "", avec_rag, False)["verdict"]

        for k in ["A", "B", "C", "D"]:
            ligne[f"source_{k}"] = sources[k]
            ligne[f"verdict_{k}"] = tester_contrefactuel(article, sources[k], avec_rag, False)["verdict"]

        lignes.append(ligne)
        print(f"[{i}/{len(echantillon)}] OK")

    with open(chemin_sortie, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=colonnes)
        w.writeheader()
        w.writerows(lignes)

In [ ]:
# execution prompt francais
MODS = ["llama3.2:1b", "llama3.2:3b", "gemma3:4b", "qwen2.5:7b","phi4-mini"]

for m in MODS:
    nom = m.replace(":", "_")

    for avec_rag in (True, False):
        mode = "RAG" if avec_rag else "WRAG"
        print("====================", mode, ":", m, "====================")

        for num, sources in enumerate(tirage, 1):
            fichier = chemin_out + f"tirage{num}_{nom}_{mode}.csv"

            if os.path.exists(fichier):
                print(f"Tirage {num} déjà fait, on saute")
                continue

            print(f"Tirage {num} -> {sources}")
            lancer_test_tirage(echantillon=articles,
                               chemin_sortie=fichier,
                               sources=sources,
                               avec_rag=avec_rag)

In [ ]:
# execution prompt aanglais 
PREFIXE = "en_2000_carc"   # change ce nom à chaque nouvelle expérience
MODS = ["llama3.2:3b", "gemma3:4b", "qwen2.5:7b", "phi4-mini"]

for m in MODS:
    nom = m.replace(":", "_")

    for avec_rag in (True, False):
        mode = "RAG" if avec_rag else "WRAG"
        print("====================", mode, ":", m, "====================")

        for num, sources in enumerate(tirage[:3], 1):
            fichier = chemin_out + f"{PREFIXE}_tirage{num}_{nom}_{mode}.csv"

            if os.path.exists(fichier):
                print(f"Tirage {num} déjà fait, on saute")
                continue

            print(f"Tirage {num} -> {sources}")
            lancer_test_tirage(echantillon=articles,
                               chemin_sortie=fichier,
                               sources=sources,
                               avec_rag=avec_rag)

In [ ]:
# demande aux modeles sil connaissent nos sources choisies dans nos tirages

tirage_chemin = "data/tirages_7.csv"


liste_domaines = ["fool.com", "buzzfeednews.com", "latimes.com", "statista.com",
                  "blogspot.com", "bgr.com", "cbs8.com", "theweek.co.uk",
                  "ibtimes.co.in", "metro.news", "dailybuzzlive.com"]


with open(tirage_chemin, newline="", encoding="utf-8") as f:
    for ligne in csv.DictReader(f):
        for cat in "ABCD":
            if ligne[cat] not in liste_domaines:
                liste_domaines.append(ligne[cat])

print(len(liste_domaines), "domaines")


MODS = ["llama3.2:1b", "llama3.2:3b", "gemma3:4b", "qwen2.5:7b", "phi4-mini"]

def question_domaine(domaine):
    return (
        f"Connais-tu le média {domaine} ?\n\n"
        "Réponds exactement dans ce format :\n"
        "Connaissance : oui ou non\n"
        f"Description : une phrase qui décrit le média {domaine}, ou « aucune » si tu ne le connais pas\n"
        "Fiabilité : fiable, peu fiable, incertain ou inconnu"
    )

for modele in MODS:
    nom = modele.replace(":", "_")
    chemin = chemin_out + f"connaissance_{nom}.csv"

    with open(chemin, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["modele", "domaine", "reponse"])

        for domaine in liste_domaines:
            rep = ollama.chat(
                model=modele,
                messages=[{"role": "user", "content": question_domaine(domaine)}],
                options={"temperature": 0}
            )
            w.writerow([modele, domaine, rep["message"]["content"].strip()])
            f.flush()

    print("OK :", chemin)

FIN

In [ ]:
def lancer_test_contrefactuel(echantillon, chemin_sortie, seed, avec_rag, creer_preuves):
    random.seed(seed)

    # 4 sources tirees, utilisees pour tous les articles
    sources = {
        "A": random.choice(catA)["domaine_principal"],
        "B": random.choice(catB)["domaine_principal"],
        "C": random.choice(catC)["domaine_principal"],
        "D": random.choice(catD)["domaine_principal"],
    }
    print(f"Seed {seed} -> sources : {sources}")

    colonnes = ["id", "vraie_source", "vrai_label",
                "source_A", "verdict_A", "source_B", "verdict_B",
                "source_C", "verdict_C", "source_D", "verdict_D",
                "verdict_S"]

    lignes = []
    for i, article in enumerate(echantillon, 1):
        ligne = {
            "id": article.get("id"),
            "vraie_source": article.get("domaine", "source inconnue"),
            "vrai_label": article.get("label"),
        }
        # sans source cree les preuves RAG 
        ligne["verdict_S"] = tester_contrefactuel(article, "", avec_rag, creer_preuves)["verdict"]

        # les 4 meme sources pour tous les articles
        for k in ["A", "B", "C", "D"]:
            ligne[f"source_{k}"] = sources[k]
            ligne[f"verdict_{k}"] = tester_contrefactuel(article, sources[k], avec_rag, False)["verdict"]

        lignes.append(ligne)
        print(f"[{i}/{len(echantillon)}] OK")

    with open(chemin_sortie, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=colonnes)
        w.writeheader()
        w.writerows(lignes)

In [ ]:

MODELES1 = ["gemma3:4b", "gemma3:1b", "gemma3:12b",
           "qwen2.5:1.5b", "qwen2.5:7b", "qwen2.5:14b",
           "llama3.2:1b", "llama3.2:3b", "phi4-mini"]

MODS = ["llama3.2:1b", "llama3.2:3b", "gemma3:4b", "qwen2.5:7b","phi4-mini"]
SEEDS = [1, 2, 3]

preuve = True # True seulement pour le tout premier passage avec RAG

for m in MODS:
    nom = m.replace(":", "_")

    for avec_rag in (True, False):
        mode = "RAG" if avec_rag else "WRAG"
        print("====================", mode, ":", m, "====================")

        for seed in SEEDS:
            lancer_test_contrefactuel(echantillon=articles,
                                      chemin_sortie=chemin_out + f"seed{seed}_{nom}_{mode}.csv",
                                      seed=seed,
                                      avec_rag=avec_rag,
                                      creer_preuves=preuve)

            preuve = False